# Lecture 4 Demo: Classical ML Refresher Through a Quantum Lens

Computational companion notebook for Lecture 4.

This notebook makes three distinctions explicit:

1. the manual XOR lift used to make the geometry visible,
2. the Gram matrix built from an explicit feature matrix,
3. the polynomial kernel as a different implicit feature map with its own geometry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)

print(f"Qiskit version: {__import__('qiskit').__version__}")

## Demo 1. Matrix-form linear/logistic baseline

We explicitly compute $z = Xw + b\mathbf{1}$ and then apply a sigmoid.

In [ ]:
X_demo = np.array([
    [1.0, 0.5],
    [0.2, 1.2],
    [1.5, 1.0],
    [0.4, 0.1],
])
w = np.array([1.1, -0.8])
b = -0.1

z = X_demo @ w + b
p = 1.0 / (1.0 + np.exp(-z))

print("z:", np.round(z, 4))
print("sigmoid(z):", np.round(p, 4))

## Demo 2. Explicit XOR lift and separating score

We start with the manual feature map
\[
\psi(x_1,x_2)=(x_1,x_2,x_1x_2).
\]

This is the geometric lift used on the slides to make XOR linearly separable in a visible 3D space.

In [ ]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
], dtype=float)
y_xor = np.array([0, 1, 1, 0])

Psi = np.column_stack([
    X_xor[:, 0],
    X_xor[:, 1],
    X_xor[:, 0] * X_xor[:, 1],
])
score = Psi[:, 0] + Psi[:, 1] - 2 * Psi[:, 2] - 0.5

print("Input points:\n", X_xor)
print("Lifted vectors psi(x):\n", Psi)
print("Separating score s = x1 + x2 - 2*x3 - 0.5:", score)

fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="coolwarm", s=80)
for idx, point in enumerate(X_xor):
    ax1.text(point[0] + 0.03, point[1] + 0.03, f"x{idx + 1}")
ax1.set_title("XOR in the original plane")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(Psi[:, 0], Psi[:, 1], Psi[:, 2], c=y_xor, cmap="coolwarm", s=80)
for idx, point in enumerate(Psi):
    ax2.text(point[0] + 0.03, point[1] + 0.03, point[2] + 0.03, f"x{idx + 1}")
ax2.set_title("Explicit lift psi(x)")
ax2.set_xlabel("x1")
ax2.set_ylabel("x2")
ax2.set_zlabel("x1*x2")

plt.tight_layout()
plt.show()

## Demo 3. Gram matrix from an explicit lift and from a polynomial kernel

The manual XOR lift $\psi(x)$ and the degree-2 polynomial kernel are related ideas, but they are not the same feature map.

For the polynomial kernel
\[
K_{\mathrm{poly}}(x,y)=(x^\top y + 1)^2,
\]
one valid explicit feature map is
\[
\phi_{\mathrm{poly}}(x_1,x_2)=
\left(1, \sqrt{2}x_1, \sqrt{2}x_2, x_1^2, \sqrt{2}x_1x_2, x_2^2\right).
\]

We compute both Gram matrices side by side.

In [ ]:
def phi_poly(X):
    x1 = X[:, 0]
    x2 = X[:, 1]
    return np.column_stack([
        np.ones(len(X)),
        np.sqrt(2.0) * x1,
        np.sqrt(2.0) * x2,
        x1 ** 2,
        np.sqrt(2.0) * x1 * x2,
        x2 ** 2,
    ])

G_explicit = Psi @ Psi.T
Phi_poly = phi_poly(X_xor)
G_poly_from_features = Phi_poly @ Phi_poly.T
G_poly_from_kernel = (X_xor @ X_xor.T + 1.0) ** 2

print("Gram matrix from the explicit XOR lift psi(x):\n", G_explicit)
print()
print("Polynomial-kernel Gram matrix from explicit phi_poly(x):\n", G_poly_from_features)
print()
print("Polynomial-kernel Gram matrix from K(x,y) = (x^T y + 1)^2:\n", G_poly_from_kernel)
print()
print("Do the two polynomial-kernel constructions agree?", np.allclose(G_poly_from_features, G_poly_from_kernel))

## Demo 4. Recover feature-space distance from kernel values

Once we know pairwise kernel values, we can recover squared distances in the implicit feature space via
\[
\|\phi(x)-\phi(y)\|^2 = K(x,x) + K(y,y) - 2K(x,y).
\]

We verify this explicitly for the polynomial-kernel feature map above.

In [ ]:
i = 1  # x^(2) = (0, 1)
j = 2  # x^(3) = (1, 0)

x_i = X_xor[i : i + 1]
x_j = X_xor[j : j + 1]
phi_i = phi_poly(x_i)[0]
phi_j = phi_poly(x_j)[0]

k_ii = float((x_i @ x_i.T + 1.0) ** 2)
k_jj = float((x_j @ x_j.T + 1.0) ** 2)
k_ij = float((x_i @ x_j.T + 1.0) ** 2)

d2_from_kernel = k_ii + k_jj - 2 * k_ij
d2_from_features = np.sum((phi_i - phi_j) ** 2)

print("x_i:", x_i[0])
print("x_j:", x_j[0])
print(f"K(x_i, x_i) = {k_ii:.1f}")
print(f"K(x_j, x_j) = {k_jj:.1f}")
print(f"K(x_i, x_j) = {k_ij:.1f}")
print(f"Distance^2 from kernel values:   {d2_from_kernel:.1f}")
print(f"Distance^2 from explicit phi(x): {d2_from_features:.1f}")

## Demo 5. Linear, polynomial, and RBF baselines on nonlinear data

Before introducing any quantum feature map, we should benchmark strong classical baselines under the same preprocessing and split.

We use `make_moons`, which is deliberately nonlinear.

In [ ]:
X_moons, y_moons = make_moons(n_samples=220, noise=0.20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42, stratify=y_moons
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svc_lin = SVC(kernel="linear", C=1.0)
svc_poly = SVC(kernel="poly", C=1.0, degree=3, gamma=1.0, coef0=1.0)
svc_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)

svc_lin.fit(X_train_s, y_train)
svc_poly.fit(X_train_s, y_train)
svc_rbf.fit(X_train_s, y_train)

acc_lin = accuracy_score(y_test, svc_lin.predict(X_test_s))
acc_poly = accuracy_score(y_test, svc_poly.predict(X_test_s))
acc_rbf = accuracy_score(y_test, svc_rbf.predict(X_test_s))

print(f"Linear SVM accuracy:      {acc_lin:.3f}")
print(f"Polynomial SVM accuracy:  {acc_poly:.3f}")
print(f"RBF SVM accuracy:         {acc_rbf:.3f}")

## Demo 6. One-qubit overlap preview: the bridge to Lecture 5

Every qubit starts in $\lvert 0\rangle$ by default.

For a one-qubit encoding
\[
\lvert\phi(x)\rangle = R_y(x)\lvert 0\rangle,
\]
we can compare the exact statevector overlap with the closed-form expression
\[
K_Q(x,y)=\left|\langle\phi(x)\vert\phi(y)\rangle\right|^2=\cos^2\left(\frac{x-y}{2}\right).
\]

In [ ]:
def ry_state(theta: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    return Statevector.from_instruction(qc).data

x = 0.9
y = 1.4

state_x = ry_state(x)
state_y = ry_state(y)
overlap = np.vdot(state_x, state_y)
kernel_exact = float(np.abs(overlap) ** 2)
kernel_formula = float(np.cos((x - y) / 2) ** 2)

print(f"<phi(x)|phi(y)> = {overlap:.6f}")
print(f"Kernel from statevectors: {kernel_exact:.6f}")
print(f"Kernel from closed form:  {kernel_formula:.6f}")